# ROI Enhancement Experiments
CLAHE, Top-Hat, Opening, Closing — visual and histogram comparisons.

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from enhancement_roi.clahe import CLAHEEnhancer
from enhancement_roi.morphological import MorphologicalOps
from enhancement_roi.roi_pipeline import ROIEnhancementPipeline

logging.basicConfig(level=logging.INFO)
PROJECT_ROOT = Path('..').resolve()
OUTPUT_DIR = PROJECT_ROOT / 'output' / '04_roi_enhancement'
PREPROC_DIR = PROJECT_ROOT / 'output' / '01_preprocessing' / 'normalized'

In [ ]:
sample_paths = sorted(PREPROC_DIR.glob('*.png'))
assert len(sample_paths) > 0, 'Run preprocessing notebook first!'
sample = cv2.imread(str(sample_paths[0]), cv2.IMREAD_GRAYSCALE)
print(f'Sample: {sample.shape}')

In [ ]:
# CLAHE parameter sweep
clahe = CLAHEEnhancer()
clip_limits = [1.0, 2.0, 3.0, 5.0, 8.0]
tile_sizes = [(4, 4), (8, 8), (16, 16)]

fig, axes = plt.subplots(len(tile_sizes), len(clip_limits), figsize=(18, 10))
for row, tg in enumerate(tile_sizes):
    for col, cl in enumerate(clip_limits):
        out = clahe.apply_with_params(sample, cl, tg)
        axes[row, col].imshow(out, cmap='gray')
        axes[row, col].set_title(f'cl={cl} tg={tg[0]}', fontsize=7)
        axes[row, col].axis('off')
plt.suptitle('CLAHE Parameter Sweep', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'clahe_param_sweep.png', dpi=150)
plt.show()

In [ ]:
# Pipeline comparison: all four modes
pipe = ROIEnhancementPipeline(clip_limit=2.0, tile_grid_size=(8,8), morph_kernel_size=15)
all_results = pipe.apply_all(sample)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (name, img) in zip(axes, all_results.items()):
    ax.imshow(img, cmap='gray')
    ax.set_title(name, fontsize=9)
    ax.axis('off')
plt.suptitle('ROI Enhancement Pipeline Comparison', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pipeline_comparison.png', dpi=150)
plt.show()

In [ ]:
# Histogram comparison for all pipeline modes
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
colors = ['gray', 'steelblue', 'darkorange', 'green', 'crimson']
for ax, (name, img), color in zip(axes, all_results.items(), colors):
    ax.hist(img.ravel(), bins=64, color=color, alpha=0.75)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('Pixel value')
plt.suptitle('Histogram Comparison — Pipeline Modes', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'histogram_comparison.png', dpi=150)
plt.show()

In [ ]:
# ROI detail comparison: crop center region
h, w = sample.shape
roi = (h//4, h*3//4, w//4, w*3//4)  # (y0, y1, x0, x1)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (name, img) in zip(axes, all_results.items()):
    crop = img[roi[0]:roi[1], roi[2]:roi[3]]
    ax.imshow(crop, cmap='gray')
    ax.set_title(f'{name}\n(ROI crop)', fontsize=9)
    ax.axis('off')
plt.suptitle('ROI Detail Comparison (Center Crop)', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roi_detail_comparison.png', dpi=150)
plt.show()

In [ ]:
# Morphological experiments
morph = MorphologicalOps()
kernel_sizes = [5, 10, 15, 25]

fig, axes = plt.subplots(3, len(kernel_sizes), figsize=(16, 12))
ops = [('Top-Hat', morph.top_hat), ('Opening', morph.opening), ('Closing', morph.closing)]
for row, (op_name, op_fn) in enumerate(ops):
    for col, k in enumerate(kernel_sizes):
        out = op_fn(sample, k)
        axes[row, col].imshow(out, cmap='gray')
        axes[row, col].set_title(f'{op_name} k={k}', fontsize=8)
        axes[row, col].axis('off')
plt.suptitle('Morphological Operations Parameter Sweep', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'morphological_sweep.png', dpi=150)
plt.show()

In [ ]:
# Batch process all images
all_paths = sorted(PREPROC_DIR.glob('*.png'))
print(f'Processing {len(all_paths)} images...')
saved = pipe.batch_process(all_paths, OUTPUT_DIR)
for mode, paths in saved.items():
    print(f'  {mode}: {len(paths)} images')